# Laboratorio 02 â€” JOINs en SQL sobre tu propio dataset

**Semana:** 03 | **Actividad de referencia:** Actividad 02  
**Modalidad:** Individual | **Entorno:** Databricks (Unity Catalog)

---

## Instrucciones generales

Aplica los tipos de JOIN y subconsultas de la Actividad 02 sobre **dos o mÃ¡s tablas de tu elecciÃ³n** en Databricks. Si tu dataset es una sola tabla, diseÃ±a un escenario donde puedas dividirla en dos tablas que tengan sentido relacionar, o agrega una tabla de referencia (p.ej., un catÃ¡logo de categorÃ­as).

## Parte 1 â€” DescripciÃ³n del escenario

1. **Tabla A:** nombre, fuente, quÃ© representa cada fila, columna(s) que usarÃ¡s para el JOIN.
2. **Tabla B:** nombre, fuente, quÃ© representa cada fila, columna(s) que usarÃ¡s para el JOIN.
3. **RelaciÃ³n esperada:** Â¿Es 1:1, 1:N, N:M? Â¿Por quÃ© elegiste estas tablas?
4. **Preguntas de negocio:** Al menos 3 preguntas que requieran combinar las tablas.

**Escribe tu respuesta aquÃ­:**

## Parte 2 â€” Cargar las tablas como Delta

In [ ]:
VOL     = "/Volumes/workspace/default/week_3"  # ajusta si usas otro volumen
TABLA_A = "workspace.default.lab03_02_tabla_a"
TABLA_B = "workspace.default.lab03_02_tabla_b"

# Carga Tabla A
df_a = spark.read.format("csv").option("header", True).option("inferSchema", True) \
    .load(f"{VOL}/archivo_a.csv")  # reemplaza con el nombre real
df_a.write.format("delta").mode("overwrite").saveAsTable(TABLA_A)
print(f"âœ“ Tabla A: {TABLA_A} â€” {df_a.count():,} filas x {len(df_a.columns)} columnas")

# Carga Tabla B
df_b = spark.read.format("csv").option("header", True).option("inferSchema", True) \
    .load(f"{VOL}/archivo_b.csv")  # reemplaza con el nombre real
df_b.write.format("delta").mode("overwrite").saveAsTable(TABLA_B)
print(f"âœ“ Tabla B: {TABLA_B} â€” {df_b.count():,} filas x {len(df_b.columns)} columnas")

## Parte 3 â€” Perfil tÃ©cnico de las tablas

In [ ]:
# Schema de Tabla A
spark.sql(f"DESCRIBE TABLE EXTENDED {TABLA_A}").show(20, truncate=False)

In [ ]:
# Schema de Tabla B
spark.sql(f"DESCRIBE TABLE EXTENDED {TABLA_B}").show(20, truncate=False)

In [ ]:
# DiagnÃ³stico de duplicados en la clave de JOIN â€” Tabla A
# Reemplaza 'clave_join_a' con el nombre real de la columna
spark.sql(f"""
    SELECT clave_join_a, COUNT(*) AS apariciones
    FROM {TABLA_A}
    GROUP BY clave_join_a
    HAVING COUNT(*) > 1
    ORDER BY apariciones DESC
    LIMIT 10
""").show(truncate=False)

**ObservaciÃ³n clave de JOIN:** Â¿Hay duplicados en la clave? Â¿CÃ³mo afecta esto al nÃºmero de filas del resultado del JOIN?

In [ ]:
# DiagnÃ³stico de duplicados en la clave de JOIN â€” Tabla B
spark.sql(f"""
    SELECT clave_join_b, COUNT(*) AS apariciones
    FROM {TABLA_B}
    GROUP BY clave_join_b
    HAVING COUNT(*) > 1
    ORDER BY apariciones DESC
    LIMIT 10
""").show(truncate=False)

## Parte 4 â€” Aplicar JOINs de la Actividad 02

Para cada tipo de JOIN muestra tambiÃ©n el nÃºmero de filas resultantes y explica la diferencia en markdown.

In [ ]:
# INNER JOIN: solo registros que coinciden en ambas tablas
resultado_inner = spark.sql(f"""
    SELECT a.*, b.*  -- limita las columnas que necesites
    FROM {TABLA_A} a
    INNER JOIN {TABLA_B} b
        ON a.clave_join_a = b.clave_join_b
    LIMIT 20
""")
resultado_inner.show(truncate=False)
print(f"Filas INNER JOIN: {resultado_inner.count():,}")

**AnÃ¡lisis INNER JOIN:** Â¿CuÃ¡ntos registros de la Tabla A no encontraron coincidencia? Â¿Eso representa un problema de calidad?

In [ ]:
# LEFT JOIN: todos los de Tabla A, coincidentes o no con Tabla B
resultado_left = spark.sql(f"""
    SELECT a.*, b.clave_join_b  -- aÃ±ade las columnas de B que necesites
    FROM {TABLA_A} a
    LEFT JOIN {TABLA_B} b
        ON a.clave_join_a = b.clave_join_b
""")
print(f"Filas LEFT JOIN (total):     {resultado_left.count():,}")
print(f"Filas sin match en B: {resultado_left.filter('clave_join_b IS NULL').count():,}")

**AnÃ¡lisis LEFT JOIN:** Los registros con `clave_join_b IS NULL` son los que no tienen correspondencia. Â¿QuÃ© significan en el contexto de negocio?

In [ ]:
# FULL OUTER JOIN: todos los de A y B aunque no coincidan
resultado_full = spark.sql(f"""
    SELECT
        COALESCE(a.clave_join_a, b.clave_join_b) AS clave,
        CASE WHEN a.clave_join_a IS NULL THEN 'Solo en B'
             WHEN b.clave_join_b IS NULL THEN 'Solo en A'
             ELSE 'En ambas' END AS origen
    FROM {TABLA_A} a
    FULL OUTER JOIN {TABLA_B} b
        ON a.clave_join_a = b.clave_join_b
""")
resultado_full.groupBy("origen").count().show()

**AnÃ¡lisis FULL OUTER JOIN:** Â¿CuÃ¡ntos registros son exclusivos de cada tabla? Â¿QuÃ© indica eso sobre la integridad referencial?

In [ ]:
# Subconsulta correlacionada: obtener el mÃ¡ximo/mÃ­nimo de B para cada registro de A
spark.sql(f"""
    SELECT
        a.*,
        (
            SELECT COUNT(*)
            FROM {TABLA_B} b
            WHERE b.clave_join_b = a.clave_join_a
        ) AS conteo_en_b
    FROM {TABLA_A} a
    ORDER BY conteo_en_b DESC
    LIMIT 15
""").show(truncate=False)

**AnÃ¡lisis subconsulta:** Â¿QuÃ© ventaja tiene la subconsulta sobre un JOIN + GROUP BY en este caso? Â¿CuÃ¡ndo podrÃ­a ser menos eficiente?

## Parte 5 â€” Preguntas de negocio con JOINs

Responde las 3 preguntas de negocio planteadas en la Parte 1 usando los JOINs adecuados.

In [ ]:
# Pregunta de negocio 1  (usa INNER, LEFT, FULL OUTER o subconsulta segÃºn convenga)
spark.sql(f"""

""").show(truncate=False)

**ConclusiÃ³n pregunta 1:**

In [ ]:
# Pregunta de negocio 2
spark.sql(f"""

""").show(truncate=False)

**ConclusiÃ³n pregunta 2:**

In [ ]:
# Pregunta de negocio 3
spark.sql(f"""

""").show(truncate=False)

**ConclusiÃ³n pregunta 3:**

## Parte 6 â€” ReflexiÃ³n final

1. Â¿QuÃ© tipo de JOIN perdiÃ³ mÃ¡s registros? Â¿Era lo esperado?
2. Â¿El nÃºmero de filas del INNER JOIN fue mayor, igual o menor a cualquiera de las tablas originales? Â¿Por quÃ©?
3. Â¿En quÃ© situaciÃ³n de tu dominio elegirÃ­as un FULL OUTER JOIN sobre un LEFT JOIN?
4. Â¿CÃ³mo expresarÃ­as la subconsulta correlacionada usando PySpark? Â¿SerÃ­a mÃ¡s sencillo o mÃ¡s complejo?

---

## Entrega en Git

```bash
# Copia el template a tu carpeta (solo la primera vez)
# cp semana_03/laboratorios/lab_02_sql_joins.ipynb semana_03/laboratorios/<tu-nombre>/lab_02_sql_joins.ipynb

git add semana_03/laboratorios/<tu-nombre>/lab_02_sql_joins.ipynb
git commit -m "lab: semana03 lab02 sql joins <nombre-dataset> - <tu-nombre>"
git push origin develop
```